In [1]:
import pandas as pd, numpy as np, xgboost as xgb, sklearn
print(pd.__version__, np.__version__, xgb.__version__, sklearn.__version__)

3.1.0 2.4.6 3.4.2 1.9.1


In [1]:
# ============================================================
# CELL 1: Setup (imports, project paths, file check)
# What it does:
#   - imports the libraries we need for cleaning
#   - defines folder paths relative to the notebook location
#   - creates data/processed and outputs if they don't exist
#   - lists every file under data/ so we can confirm the exact
#     CSV names and where they live
# ============================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

# Notebook sits in rainfall_analysis/notebooks, so the project root is one level up
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = ROOT / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__)
print("\nFiles under data/:")
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(DATA_DIR)}  ({p.stat().st_size / 1024:.1f} KB)")

Project root: d:\rainfall_analysis
pandas 3.1.0 | numpy 2.4.6

Files under data/:
  raw\all_stations.csv  (567.3 KB)
  raw\lamachaur_only.csv  (422.8 KB)


In [2]:
# ============================================================
# CELL 2: Peek at the raw files
# What it does:
#   - prints the first and last raw text lines of each CSV, to catch
#     header quirks, units rows, delimiters or footer junk
#   - loads both files with every column as text (dtype=str)
#   - prints shape, column names and the first rows
#   - for each column, reports blanks and any values that are not plain
#     numbers (e.g. -999, T, M, ---), which reveals the missing-data codes
# ============================================================
RAW_DIR = DATA_DIR / "raw"
ALL_FILE = RAW_DIR / "all_stations.csv"
LAM_FILE = RAW_DIR / "lamachaur_only.csv"

# --- 1) raw text look ---
for f in (ALL_FILE, LAM_FILE):
    print("=" * 70)
    print(f.name)
    with open(f, "r", encoding="utf-8-sig", errors="replace") as fh:
        lines = fh.readlines()
    print("total lines:", len(lines))
    print("--- first 6 raw lines ---")
    for line in lines[:6]:
        print(repr(line))
    print("--- last 3 raw lines ---")
    for line in lines[-3:]:
        print(repr(line))

# --- 2) load everything as text ---
df_all = pd.read_csv(ALL_FILE, dtype=str, encoding="utf-8-sig")
df_lam = pd.read_csv(LAM_FILE, dtype=str, encoding="utf-8-sig")

# --- 3) structure + non-numeric tokens per column ---
def inspect(df, name):
    print("\n" + "=" * 70)
    print(f"{name}: shape = {df.shape}")
    print("columns:", list(df.columns))
    print(df.head(8))
    print("\nPer-column summary (blanks, non-numeric tokens):")
    for col in df.columns:
        s = df[col]
        blanks = s.isna().sum() + (s.str.strip() == "").sum()
        num = pd.to_numeric(s, errors="coerce")
        bad = s[num.isna() & s.notna() & (s.str.strip() != "")].unique()
        print(f"  {col!r}: blanks={blanks}, non-numeric unique={len(bad)}, sample={list(bad[:6])}")

inspect(df_all, "all_stations")
inspect(df_lam, "lamachaur_only")

all_stations.csv
total lines: 14976
--- first 6 raw lines ---
'Time,Pokhara Airport,Kunchha,Bandipur,Lumle,Damauli,Sikles,Lamachaur\n'
'01/01/1983 03:00:00,0,0,0,0,,0,\n'
'01/02/1983 03:00:00,1.6,0,0,0,,10,\n'
'01/03/1983 03:00:00,0.4,0,0,0.5,,12.2,\n'
'01/04/1983 03:00:00,0,0,0,0,,0,\n'
'01/05/1983 03:00:00,0,0,0,0,,0,\n'
--- last 3 raw lines ---
'12/29/2023 03:00:00,0,0,0,0,0,0,0\n'
'12/30/2023 03:00:00,0,0,0,0,0,0,0\n'
'12/31/2023 03:00:00,0,0,0,0,0,0,0'
lamachaur_only.csv
total lines: 18233
--- first 6 raw lines ---
'Time,Lamachaur\n'
'01/01/1972 03:00:00,0\n'
'01/02/1972 03:00:00,0\n'
'01/03/1972 03:00:00,0\n'
'01/04/1972 03:00:00,0\n'
'01/05/1972 03:00:00,0\n'
--- last 3 raw lines ---
'08/14/2023 03:00:00,84\n'
'08/15/2023 03:00:00,0.01\n'
'08/16/2023 03:00:00,90'

all_stations: shape = (14975, 8)
columns: ['Time', 'Pokhara Airport', 'Kunchha', 'Bandipur', 'Lumle', 'Damauli', 'Sikles', 'Lamachaur']
                  Time Pokhara Airport Kunchha Bandipur Lumle Damauli Sikles Lamac

In [3]:
# ============================================================
# CELL 3: Parse dates and check the time axis
# What it does:
#   - converts 'Time' to real datetimes (month/day/year) and confirms
#     every timestamp is 03:00:00
#   - builds a plain daily 'date' column
#   - for each file: date range, duplicate dates, and missing days
#     compared with a full daily calendar, plus the longest gaps
#   - converts station columns from text to numbers
#   - for all_stations: valid-day count and first/last valid date per
#     station (shows where Lamachaur's 365 values sit)
# ============================================================
TIME_FMT = "%m/%d/%Y %H:%M:%S"

all_p = df_all.copy()
lam_p = df_lam.copy()
for df in (all_p, lam_p):
    df["Time"] = pd.to_datetime(df["Time"], format=TIME_FMT)
    df["date"] = df["Time"].dt.normalize()

print("Clock times, all_stations:  ", all_p["Time"].dt.strftime("%H:%M:%S").unique())
print("Clock times, lamachaur_only:", lam_p["Time"].dt.strftime("%H:%M:%S").unique())

# text -> numbers
stations = [c for c in df_all.columns if c != "Time"]
for c in stations:
    all_p[c] = pd.to_numeric(all_p[c], errors="coerce")
lam_p["Lamachaur"] = pd.to_numeric(lam_p["Lamachaur"], errors="coerce")

def time_axis_report(df, name):
    full = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    missing = full.difference(pd.DatetimeIndex(df["date"].unique()))
    print("\n" + "=" * 70)
    print(name)
    print("range:", df["date"].min().date(), "->", df["date"].max().date())
    print("rows:", len(df), "| expected daily rows:", len(full))
    print("duplicate dates:", int(df["date"].duplicated().sum()))
    print("sorted ascending:", df["date"].is_monotonic_increasing)
    print("missing dates (absent rows):", len(missing))
    if len(missing):
        s = pd.Series(missing)
        grp = (s.diff() != pd.Timedelta(days=1)).cumsum()
        gaps = s.groupby(grp).agg(start="min", end="max", days="count")
        gaps = gaps.sort_values("days", ascending=False)
        print("number of separate gaps:", len(gaps))
        print("longest gaps:")
        print(gaps.head(10).to_string(index=False))

time_axis_report(all_p, "all_stations")
time_axis_report(lam_p, "lamachaur_only")

# coverage per station in all_stations
rows = []
for c in stations:
    v = all_p.loc[all_p[c].notna(), "date"]
    span = (v.max() - v.min()).days + 1 if len(v) else 0
    rows.append({
        "station": c,
        "valid_days": len(v),
        "first_valid": v.min().date() if len(v) else None,
        "last_valid": v.max().date() if len(v) else None,
        "blank_within_span": span - len(v),
    })
print("\n" + "=" * 70)
print("Coverage per station in all_stations:")
print(pd.DataFrame(rows).to_string(index=False))
print("\nlamachaur_only NaN count:", int(lam_p["Lamachaur"].isna().sum()))

Clock times, all_stations:   <ArrowStringArray>
['03:00:00']
Length: 1, dtype: str
Clock times, lamachaur_only: <ArrowStringArray>
['03:00:00']
Length: 1, dtype: str

all_stations
range: 1983-01-01 -> 2023-12-31
rows: 14975 | expected daily rows: 14975
duplicate dates: 0
sorted ascending: True
missing dates (absent rows): 0

lamachaur_only
range: 1972-01-01 -> 2023-08-16
rows: 18232 | expected daily rows: 18856
duplicate dates: 0
sorted ascending: True
missing dates (absent rows): 624
number of separate gaps: 7
longest gaps:
     start        end  days
2005-02-01 2005-12-31   334
1986-10-01 1987-03-02   153
1998-11-01 1998-12-31    61
2020-12-01 2020-12-31    31
2001-06-01 2001-06-30    30
2023-04-18 2023-04-30    13
2009-12-30 2009-12-31     2

Coverage per station in all_stations:
        station  valid_days first_valid last_valid  blank_within_span
Pokhara Airport       14975  1983-01-01 2023-12-31                  0
        Kunchha       14909  1983-01-01 2023-12-31                

In [4]:
# ============================================================
# CELL 4: Check the rainfall values
# What it does:
#   - builds date-indexed daily series; lamachaur_only is reindexed to
#     a full calendar so its 624 absent days show up as NaN
#   - per station: min/max/mean, % zero days, negatives, sentinel-like
#     codes (-999, 9999 ...), trace values (0 < x < 0.1), and values
#     with more than 1 decimal
#   - 8 largest daily values with dates (spikes or typos)
#   - 5 longest runs of consecutive zeros (missing data coded as 0)
#   - annual totals table (years with fewer than 350 valid days are
#     left blank, so unusually low totals stand out)
# ============================================================
gauge_cols = [c for c in stations if c != "Lamachaur"]   # Lamachaur here has only 365 values
all_ts = all_p.set_index("date")[gauge_cols]

lam_full_idx = pd.date_range(lam_p["date"].min(), lam_p["date"].max(), freq="D")
lam_ts = lam_p.set_index("date")["Lamachaur"].reindex(lam_full_idx)
lam_ts.index.name = "date"

series_dict = {c: all_ts[c] for c in all_ts.columns}
series_dict["Lamachaur (lam_only)"] = lam_ts

SENTINELS = [-999, -99, -9999, 99, 999, 9999, 99.9, 999.9]

def zero_runs(s, top=5):
    z = s.eq(0)
    grp = (z != z.shift(fill_value=False)).cumsum()
    out = []
    for _, chunk in z.groupby(grp):
        if chunk.iloc[0]:
            out.append((len(chunk), chunk.index[0].date().isoformat(), chunk.index[-1].date().isoformat()))
    return sorted(out, reverse=True)[:top]

for name, s in series_dict.items():
    v = s.dropna()
    print("=" * 70)
    print(name)
    print(f"valid={len(v)} | NaN={int(s.isna().sum())} | min={v.min()} | max={v.max()} "
          f"| mean={v.mean():.2f} | zero days={(v == 0).mean() * 100:.1f}%")
    print("negatives:", int((v < 0).sum()),
          "| sentinel-like:", int(v.isin(SENTINELS).sum()),
          "| trace (0<x<0.1):", int(((v > 0) & (v < 0.1)).sum()),
          "| >1 decimal:", int((((v * 10).round(6) % 1) != 0).sum()))
    print("top 8:", [(d.date().isoformat(), x) for d, x in v.nlargest(8).items()])
    print("longest zero runs (days, start, end):")
    for r in zero_runs(s):
        print("   ", r)

# annual totals; years with < 350 valid days are blanked
ann = {}
for name, s in series_dict.items():
    g = s.groupby(s.index.year)
    ann[name] = g.sum(min_count=1).where(g.count() >= 350)
annual = pd.DataFrame(ann).round(0)
annual.index.name = "year"
print("\n" + "=" * 70)
print("Annual totals (mm), blank = fewer than 350 valid days")
print(annual.to_string())

Pokhara Airport
valid=14975 | NaN=0 | min=0.0 | max=357.0 | mean=10.64 | zero days=54.5%
negatives: 0 | sentinel-like: 2 | trace (0<x<0.1): 82 | >1 decimal: 82
top 8: [('2001-08-18', 357.0), ('2021-09-03', 280.0), ('2016-09-18', 274.2), ('2010-07-11', 262.1), ('2021-08-23', 260.1), ('2021-07-07', 258.2), ('2020-07-10', 250.9), ('2022-08-01', 249.0)]
longest zero runs (days, start, end):
    (158, '2008-10-10', '2009-03-16')
    (107, '2020-10-21', '2021-02-04')
    (97, '2016-10-18', '2017-01-22')
    (86, '2022-11-19', '2023-02-12')
    (84, '2012-10-26', '2013-01-17')
Kunchha
valid=14909 | NaN=66 | min=0.0 | max=215.8 | mean=7.00 | zero days=72.6%
negatives: 0 | sentinel-like: 1 | trace (0<x<0.1): 0 | >1 decimal: 0
top 8: [('2000-06-24', 215.8), ('2022-06-06', 208.6), ('2021-07-03', 195.3), ('2014-09-11', 187.2), ('2010-04-17', 178.2), ('2002-07-29', 177.5), ('2015-07-30', 177.5), ('1990-06-10', 176.4)]
longest zero runs (days, start, end):
    (167, '2008-10-10', '2009-03-25')
    (

In [5]:
# ============================================================
# CELL 5: Suspect-value audit (cross-checked against other stations)
# What it does:
#   A) lists values that look like missing-data codes (99, 999, ...)
#      with what the other stations recorded that day
#   B) takes each station's 8 largest days and compares them with the
#      other stations' maximum over day-1 .. day+1 (flags CHECK if the
#      others recorded under 20% of it, which suggests a typo or an
#      unmatched spike)
#   C) finds zero runs of 10+ days inside Jun-Sep and the average rain
#      the other stations recorded over the same days; flags runs where
#      others averaged over 3 mm/day (or had no data), i.e. zeros that
#      are probably missing data
#   D) counts trace values (0 < x < 0.1) per year and station
# ============================================================
LAM = "Lamachaur (lam_only)"
combined = pd.concat([all_ts, lam_ts.rename(LAM)], axis=1).sort_index()
cols = list(combined.columns)

# --- A) code-like values ---
print("A) Code-like values (99, 999, ...) with other stations the same day")
for col in cols:
    hits = combined[col][combined[col].isin(SENTINELS)]
    for d, x in hits.items():
        nb = combined.loc[d, [c for c in cols if c != col]].round(1).to_dict()
        print(f"  {col} | {d.date()} | {x} | others: {nb}")

# --- B) extremes cross-check ---
rows = []
for col in cols:
    others = [c for c in cols if c != col]
    for d, x in combined[col].dropna().nlargest(8).items():
        win = combined.loc[d - pd.Timedelta(days=1): d + pd.Timedelta(days=1), others]
        nb_max = win.max().max()
        same_day = combined.loc[d, others].dropna()
        rows.append({
            "station": col, "date": d.date(), "value": x,
            "others_same_day_max": round(same_day.max(), 1) if len(same_day) else np.nan,
            "others_pm1day_max": round(nb_max, 1) if pd.notna(nb_max) else np.nan,
        })
ext = pd.DataFrame(rows)
ext["flag"] = np.where(ext["others_pm1day_max"] < 0.2 * ext["value"], "CHECK", "")
print("\nB) Top-8 days per station vs other stations")
print(ext.to_string(index=False))

# --- C) monsoon zero runs ---
def monsoon_zero_runs(col, min_len=10):
    s = combined[col]
    others = [c for c in cols if c != col]
    sm = s[s.index.month.isin([6, 7, 8, 9])]
    out = []
    for yr, sy in sm.groupby(sm.index.year):
        z = sy.eq(0)
        grp = (z != z.shift(fill_value=False)).cumsum()
        for _, ch in z.groupby(grp):
            if ch.iloc[0] and len(ch) >= min_len:
                a, b = ch.index[0], ch.index[-1]
                nb = combined.loc[a:b, others].mean(axis=1).mean()
                out.append({"start": a.date(), "end": b.date(), "days": len(ch),
                            "others_mean_mm_day": round(nb, 1) if pd.notna(nb) else np.nan})
    return pd.DataFrame(out)

print("\nC) Zero runs >= 10 days in Jun-Sep (flagged if others averaged > 3 mm/day or had no data)")
for col in cols:
    r = monsoon_zero_runs(col)
    if len(r) == 0:
        print(f"\n  {col}: no zero runs of 10+ days in Jun-Sep")
        continue
    flagged = r[(r["others_mean_mm_day"] > 3) | r["others_mean_mm_day"].isna()]
    print(f"\n  {col}: {len(r)} runs, {len(flagged)} flagged")
    if len(flagged):
        print(flagged.sort_values("days", ascending=False).head(12).to_string(index=False))

# --- D) trace values per year ---
print("\nD) Trace values (0 < x < 0.1) per year")
for col in cols:
    s = combined[col]
    tr = s[(s > 0) & (s < 0.1)]
    print(f"  {col}: {tr.groupby(tr.index.year).size().to_dict()}")

A) Code-like values (99, 999, ...) with other stations the same day
  Pokhara Airport | 1991-07-15 | 99.0 | others: {'Kunchha': 0.0, 'Bandipur': 15.1, 'Lumle': 99.0, 'Damauli': 44.0, 'Sikles': 25.4, 'Lamachaur (lam_only)': 140.5}
  Pokhara Airport | 2016-09-17 | 99.9 | others: {'Kunchha': 0.0, 'Bandipur': 2.4, 'Lumle': 81.1, 'Damauli': 0.0, 'Sikles': 48.5, 'Lamachaur (lam_only)': 82.0}
  Kunchha | 2022-07-21 | 99.0 | others: {'Pokhara Airport': 105.7, 'Bandipur': 30.0, 'Lumle': 75.5, 'Damauli': 2.7, 'Sikles': 41.7, 'Lamachaur (lam_only)': 190.8}
  Lumle | 1991-07-15 | 99.0 | others: {'Pokhara Airport': 99.0, 'Kunchha': 0.0, 'Bandipur': 15.1, 'Damauli': 44.0, 'Sikles': 25.4, 'Lamachaur (lam_only)': 140.5}
  Lumle | 2004-08-31 | 99.0 | others: {'Pokhara Airport': 59.2, 'Kunchha': 57.8, 'Bandipur': 0.0, 'Damauli': 2.2, 'Sikles': 20.0, 'Lamachaur (lam_only)': 84.0}
  Lumle | 2014-09-26 | 99.0 | others: {'Pokhara Airport': 131.5, 'Kunchha': 40.3, 'Bandipur': 0.0, 'Damauli': 0.0, 'Sikles': 5

In [6]:
# ============================================================
# CELL 6: Monthly audit - how much of the "zero" data is really missing?
# What it does:
#   A) builds monthly totals, valid-day counts and zero-day fractions
#      for all 7 series
#   B) calibration: for Jun-Sep months, counts the months with >=50/60/
#      70/80/90% zero days per station; Pokhara Airport, Lumle and
#      Sikles show how dry a real monsoon month gets
#   C) ratio check: each month's total divided by the mean total of the
#      reference stations (Pokhara Airport, Lumle, Sikles, excluding the
#      station itself), then divided by that station's typical ratio for
#      the same calendar month. A value under 0.3 means the station got
#      less than 30% of its usual share while references had >= 100 mm
#   D) prints the worst months for the four suspect stations and keeps
#      them in flag_table for the cleaning step
# Note: Lamachaur 1973-1982 cannot be audited (no reference stations
#       before 1983)
# ============================================================
REF = ["Pokhara Airport", "Lumle", "Sikles"]
SUSPECT = ["Kunchha", "Bandipur", "Damauli", LAM]
JJAS = [6, 7, 8, 9]

def monthly_stats(s):
    keys = [s.index.year, s.index.month]
    zero = (s == 0).astype(float).where(s.notna())
    out = pd.DataFrame({
        "total": s.groupby(keys).sum(min_count=1),
        "valid": s.groupby(keys).count(),
        "zero_days": zero.groupby(keys).sum(min_count=1),
    })
    out.index.names = ["year", "month"]
    out["zero_frac"] = out["zero_days"] / out["valid"]
    return out

mt = {c: monthly_stats(combined[c]) for c in cols}

# --- B) calibration table ---
print("B) Jun-Sep months by share of zero days (months with >= 25 valid days)")
rows = []
for c in cols:
    d = mt[c][mt[c]["valid"] >= 25]
    d = d[d.index.get_level_values("month").isin(JJAS)]
    row = {"station": c, "JJAS months": len(d)}
    for t in (0.5, 0.6, 0.7, 0.8, 0.9):
        row[f">={int(t * 100)}% zero"] = int((d["zero_frac"] >= t).sum())
    row["max share"] = round(d["zero_frac"].max(), 2)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

# --- C) ratio check against reference stations ---
frames = []
for c in cols:
    refs = [r for r in REF if r != c]
    ref_tot = pd.concat(
        [mt[r]["total"].where(mt[r]["valid"] >= 25) for r in refs], axis=1
    ).mean(axis=1, skipna=False)
    d = mt[c].copy()
    d["ref_total"] = ref_tot.reindex(d.index)
    ok = (d["valid"] >= 25) & (d["ref_total"] >= 100)
    ratio = (d["total"] / d["ref_total"]).where(ok)
    med = ratio.groupby(level="month").median()
    d["ratio_norm"] = ratio / med.reindex(d.index.get_level_values("month")).to_numpy()
    d["station"] = c
    frames.append(d.reset_index())

audit = pd.concat(frames, ignore_index=True)
audit["flag"] = audit["ratio_norm"] < 0.3

summ = audit.groupby("station").agg(months_compared=("ratio_norm", "count"),
                                    months_flagged=("flag", "sum")).reindex(cols)
summ["pct_flagged"] = (100 * summ["months_flagged"] / summ["months_compared"]).round(1)
print("\nC) Months flagged by the ratio check (ratio_norm < 0.3)")
print(summ.to_string())

# --- D) worst months for suspect stations ---
flag_table = audit[audit["flag"] & audit["station"].isin(SUSPECT)][
    ["station", "year", "month", "valid", "zero_days", "total", "ref_total", "ratio_norm", "zero_frac"]
].copy()
flag_table = flag_table.round(2)

for c in SUSPECT:
    t = flag_table[flag_table["station"] == c]
    print("\n" + "=" * 70)
    print(f"D) {c}: {len(t)} flagged months")
    print("flagged months per year:", t.groupby("year").size().to_dict())
    print("worst 12 (lowest ratio_norm):")
    print(t.sort_values("ratio_norm").head(12).drop(columns="station").to_string(index=False))

B) Jun-Sep months by share of zero days (months with >= 25 valid days)
             station  JJAS months  >=50% zero  >=60% zero  >=70% zero  >=80% zero  >=90% zero  max share
     Pokhara Airport          164           1           0           0           0           0       0.50
             Kunchha          164          67          41          17           5           1       0.90
            Bandipur          159          67          42          21           4           1       0.90
               Lumle          164           0           0           0           0           0       0.37
             Damauli          164          69          38          15           7           2       0.93
              Sikles          164           0           0           0           0           0       0.47
Lamachaur (lam_only)          201           7           5           5           5           5       1.00

C) Months flagged by the ratio check (ratio_norm < 0.3)
                      months_com

In [7]:
# ============================================================
# CELL 7: Build the masking plan (nothing is modified yet)
# What it does:
#   1) takes every month flagged in Cell 6 (ratio_norm < 0.3) and
#      labels it Tier A (zero share >= 0.85 -> would be masked) or
#      Tier B (flag only)
#   2) adds two manual items: Lamachaur 1972-01-01..1973-01-06 (mask)
#      and Sikles 2009-08-29 (flag)
#   3) stores everything in mask_plan (station, start, end, action,
#      reason, valid days and rainfall inside the period)
#   4) prints a summary per station, the full plan, and station-years
#      with 3+ flagged months next to the reference-station average,
#      as candidates for dropping the whole year
# ============================================================
ZERO_FRAC_MIN = 0.85
RATIO_MAX = 0.30

cand = audit[audit["flag"] & (audit["ratio_norm"] < RATIO_MAX)].copy()
cand["tier"] = np.where(cand["zero_frac"] >= ZERO_FRAC_MIN, "A_mask", "B_flag_only")

rows = []
for r in cand.itertuples():
    s0 = pd.Timestamp(year=int(r.year), month=int(r.month), day=1)
    rows.append((r.station, s0, s0 + pd.offsets.MonthEnd(0),
                 "mask" if r.tier == "A_mask" else "flag",
                 f"ratio {r.ratio_norm:.2f}, zero share {r.zero_frac:.2f}"))
rows.append((LAM, pd.Timestamp("1972-01-01"), pd.Timestamp("1973-01-06"),
             "mask", "372-day zero run; 1972 total 0 mm"))
rows.append(("Sikles", pd.Timestamp("2009-08-29"), pd.Timestamp("2009-08-29"),
             "flag", "99 mm while the other six stations read 0 to 1.3 mm"))

mask_plan = (pd.DataFrame(rows, columns=["station", "start", "end", "action", "reason"])
             .sort_values(["station", "start"]).reset_index(drop=True))
mask_plan["valid_days"] = mask_plan.apply(
    lambda r: int(combined.loc[r["start"]:r["end"], r["station"]].notna().sum()), axis=1)
mask_plan["mm_in_period"] = mask_plan.apply(
    lambda r: round(combined.loc[r["start"]:r["end"], r["station"]].sum(), 1), axis=1)
mask_plan["year"] = mask_plan["start"].dt.year

print("Summary: number of periods and valid days per station/action")
summ = (mask_plan.groupby(["station", "action"])
        .agg(periods=("start", "count"), valid_days=("valid_days", "sum"))
        .unstack(fill_value=0))
print(summ.to_string())

print("\nFull plan:")
show = mask_plan.drop(columns="year").copy()
show["start"] = show["start"].dt.date
show["end"] = show["end"].dt.date
print(show.to_string(index=False))

# station-years with 3+ flagged months
ratio_rows = mask_plan[mask_plan["reason"].str.startswith("ratio")]
cnt = ratio_rows.groupby(["station", "year"]).size()
whole = cnt[cnt >= 3]
ann_raw = combined.groupby(combined.index.year).sum(min_count=1)
ref_mean = ann_raw[REF].mean(axis=1)
print("\nStation-years with 3+ flagged months (candidates to drop the whole year):")
if len(whole) == 0:
    print("  none")
for (st, yr), n in whole.items():
    print(f"  {st} {yr}: {n} flagged months | station total = {ann_raw.loc[yr, st]:.0f} mm "
          f"| mean of Pokhara/Lumle/Sikles = {ref_mean.loc[yr]:.0f} mm")

Summary: number of periods and valid days per station/action
                     periods      valid_days     
action                  flag mask       flag mask
station                                          
Bandipur                   5   12        154  368
Damauli                    0   11          0  338
Kunchha                    0    7          0  214
Lamachaur (lam_only)       0    2          0  403
Lumle                      1    1         30   31
Pokhara Airport            1    0         30    0
Sikles                     2    0         32    0

Full plan:
             station      start        end action                                              reason  valid_days  mm_in_period
            Bandipur 1992-10-01 1992-10-31   mask                         ratio 0.00, zero share 1.00          31           0.0
            Bandipur 1995-03-01 1995-03-31   mask                         ratio 0.27, zero share 0.97          31           7.2
            Bandipur 1996-06-01 1996-06-30 

In [8]:
# ============================================================
# CELL 8: Lamachaur - compare the two sources and build one merged series
# What it does:
#   1) compares the Lamachaur column in all_stations (365 values, all of
#      2023) with lamachaur_only on the days where both have a value
#   2) prints how many days match exactly and lists any that differ
#   3) lists the days present only in all_stations
#   4) builds lam_merged: lamachaur_only is the base; days missing there
#      are filled from all_stations and the series is extended to
#      2023-12-31
#   5) prints the remaining gaps and a 2023 monthly sanity check
#      against Pokhara Airport and Lumle
# Nothing is masked here; masking happens in Cell 9.
# ============================================================
lam_a = all_p.set_index("date")["Lamachaur"].dropna()   # all_stations (2023 only)
lam_b = lam_ts.dropna()                                  # lamachaur_only

# --- 1-2) overlap comparison ---
overlap = lam_a.index.intersection(lam_b.index)
cmp_df = pd.DataFrame({"all_stations": lam_a.loc[overlap],
                       "lamachaur_only": lam_b.loc[overlap]})
cmp_df["diff"] = cmp_df["all_stations"] - cmp_df["lamachaur_only"]
same = cmp_df["diff"].abs() < 1e-9

print("Days with data | all_stations:", len(lam_a), "| lamachaur_only:", len(lam_b))
print("Overlapping days:", len(overlap))
if len(overlap):
    print("Overlap range:", overlap.min().date(), "->", overlap.max().date())
    print("Identical:", int(same.sum()), "| different:", int((~same).sum()))
    print("Max abs difference (mm):", round(cmp_df["diff"].abs().max(), 2))
    if (~same).any():
        print("\nDays that differ:")
        print(cmp_df[~same].round(2).to_string())

# --- 3) days only in all_stations ---
def gap_table(idx):
    s = pd.Series(idx)
    grp = (s.diff() != pd.Timedelta(days=1)).cumsum()
    return (s.groupby(grp).agg(start="min", end="max", days="count")
             .sort_values("days", ascending=False))

only_a = lam_a.index.difference(lam_b.index)
print("\nDays only in all_stations:", len(only_a))
if len(only_a):
    print(gap_table(only_a).to_string(index=False))

# --- 4) merged series ---
end = max(lam_ts.index.max(), lam_a.index.max())
full_idx = pd.date_range(lam_ts.index.min(), end, freq="D")
lam_merged = lam_ts.reindex(full_idx)
lam_merged.loc[only_a] = lam_a.loc[only_a]
lam_merged.index.name = "date"

nan_days = lam_merged.index[lam_merged.isna()]
print("\nlam_merged:", lam_merged.index.min().date(), "->", lam_merged.index.max().date())
print("valid days:", int(lam_merged.notna().sum()), "| NaN days:", len(nan_days))
if len(nan_days):
    print("Remaining gaps:")
    print(gap_table(nan_days).to_string(index=False))

# --- 5) 2023 sanity check ---
y = slice("2023-01-01", "2023-12-31")
lm = lam_merged.loc[y]
cb = combined.loc[y]
chk = pd.DataFrame({
    "Lamachaur (merged)": lm.groupby(lm.index.month).sum(min_count=1),
    "Pokhara Airport": cb["Pokhara Airport"].groupby(cb.index.month).sum(min_count=1),
    "Lumle": cb["Lumle"].groupby(cb.index.month).sum(min_count=1),
}).round(0)
chk.index.name = "month"
print("\n2023 monthly totals (mm):")
print(chk.to_string())

Days with data | all_stations: 365 | lamachaur_only: 18232
Overlapping days: 215
Overlap range: 2023-01-01 -> 2023-08-16
Identical: 215 | different: 0
Max abs difference (mm): 0.0

Days only in all_stations: 150
     start        end  days
2023-08-17 2023-12-31   137
2023-04-18 2023-04-30    13

lam_merged: 1972-01-01 -> 2023-12-31
valid days: 18382 | NaN days: 611
Remaining gaps:
     start        end  days
2005-02-01 2005-12-31   334
1986-10-01 1987-03-02   153
1998-11-01 1998-12-31    61
2020-12-01 2020-12-31    31
2001-06-01 2001-06-30    30
2009-12-30 2009-12-31     2

2023 monthly totals (mm):
       Lamachaur (merged)  Pokhara Airport   Lumle
month                                             
1                     0.0              0.0     0.0
2                    13.0             34.0    14.0
3                   105.0            172.0   128.0
4                    88.0            101.0    80.0
5                   224.0            236.0   205.0
6                   549.0           

In [10]:
# ============================================================
# CELL 9 (revised): Apply the masking plan, cut to 1983, save
# What it does:
#   1) builds the 7-station table on the full calendar (1972-2023)
#   2) applies every "mask" period from mask_plan plus the whole-year
#      drops in DROP_YEARS (edit below to veto), and builds the flags
#      table for "flag" periods (values kept, only marked)
#   3) saves the full-history Lamachaur series (masks applied) as
#      lamachaur_long_clean.parquet for possible later use
#   4) cuts everything to START = 1983-01-01 for the modelling table
#   5) prints per station: valid days before/after masking, days masked,
#      flagged days, years with >= 350 valid days; plus how many days
#      have all 7 stations valid at once
#   6) saves stations_clean.parquet (+ .csv), stations_flags.parquet and
#      outputs/mask_plan.csv, then reloads the parquet and checks it
# ============================================================
START = "1983-01-01"
DROP_YEARS = {"Bandipur": [1998, 2017]}   # edit or empty this dict to veto

gauge_cols = ["Pokhara Airport", "Kunchha", "Bandipur", "Lumle", "Damauli", "Sikles"]
station_order = gauge_cols + ["Lamachaur"]

# --- 1) full-history table ---
raw_full = pd.concat([combined[gauge_cols], lam_merged.rename("Lamachaur")], axis=1)
raw_full = raw_full.reindex(pd.date_range(raw_full.index.min(), raw_full.index.max(), freq="D"))
raw_full.index.name = "date"
raw_full = raw_full[station_order]

clean_full = raw_full.copy()
flags_full = pd.DataFrame(False, index=raw_full.index, columns=station_order)
name_map = {LAM: "Lamachaur"}

# --- 2) apply masks, flags and whole-year drops ---
for r in mask_plan.itertuples():
    col = name_map.get(r.station, r.station)
    if r.action == "mask":
        clean_full.loc[r.start:r.end, col] = np.nan
    else:
        flags_full.loc[r.start:r.end, col] = True

for col, years in DROP_YEARS.items():
    for y in years:
        clean_full.loc[f"{y}-01-01":f"{y}-12-31", col] = np.nan

# --- 3) long Lamachaur record, kept separately ---
lam_long = clean_full[["Lamachaur"]].dropna()
lam_long.to_parquet(PROCESSED_DIR / "lamachaur_long_clean.parquet")
print(f"Saved lamachaur_long_clean.parquet: {len(lam_long)} valid days, "
      f"{lam_long.index.min().date()} -> {lam_long.index.max().date()}")

# --- 4) cut to the modelling period ---
raw = raw_full.loc[START:]
clean = clean_full.loc[START:]
flags = flags_full.loc[START:] & clean.notna()

# --- 5) report ---
rows = []
for c in station_order:
    before, after = int(raw[c].notna().sum()), int(clean[c].notna().sum())
    yrs = clean[c].groupby(clean.index.year).count()
    rows.append({
        "station": c,
        "first_valid": clean[c].first_valid_index().date(),
        "last_valid": clean[c].last_valid_index().date(),
        "valid_before": before, "valid_after": after,
        "masked_days": before - after,
        "pct_masked": round(100 * (before - after) / before, 1),
        "flagged_days": int(flags[c].sum()),
        "years>=350d": int((yrs >= 350).sum()),
    })
print("\n" + pd.DataFrame(rows).to_string(index=False))

n_all = int(clean.notna().all(axis=1).sum())
print(f"\nCalendar days: {len(clean)} | days with all 7 stations valid: {n_all} "
      f"({100 * n_all / len(clean):.1f}%)")

ann = clean.groupby(clean.index.year).sum(min_count=1).where(
    clean.groupby(clean.index.year).count() >= 350)
print("\nBandipur annual totals after cleaning (selected years):")
print(ann.loc[[1992, 1997, 1998, 1999, 2016, 2017, 2018], "Bandipur"])

# --- 6) save + reload check ---
out_clean = PROCESSED_DIR / "stations_clean.parquet"
clean.to_parquet(out_clean)
clean.to_csv(PROCESSED_DIR / "stations_clean.csv")
flags.to_parquet(PROCESSED_DIR / "stations_flags.parquet")
mask_plan.to_csv(OUTPUT_DIR / "mask_plan.csv", index=False)

chk = pd.read_parquet(out_clean)
print("\nSaved:", out_clean.name, "| shape:", chk.shape,
      "| range:", chk.index.min().date(), "->", chk.index.max().date())
print("Reload matches in-memory table:", chk.equals(clean))

Saved lamachaur_long_clean.parquet: 17979 valid days, 1973-01-07 -> 2023-12-31

        station first_valid last_valid  valid_before  valid_after  masked_days  pct_masked  flagged_days  years>=350d
Pokhara Airport  1983-01-01 2023-12-31         14975        14975            0         0.0            30           41
        Kunchha  1983-01-01 2023-12-31         14909        14695          214         1.4             0           32
       Bandipur  1983-01-01 2023-12-31         14486        13540          946         6.5            30           29
          Lumle  1983-01-01 2023-12-31         14940        14909           31         0.2            30           39
        Damauli  1983-04-01 2023-12-31         14581        14243          338         2.3             0           26
         Sikles  1983-01-01 2023-12-31         14942        14942            0         0.0            32           40
      Lamachaur  1983-01-01 2023-12-31         14364        14333           31         0.2    